# CV面试题 11: 深度学习视觉面试题

本节涵盖计算机视觉深度学习面试中的高频考点：
- CNN 架构细节与归一化方法
- 目标检测 (YOLO/SSD/FCOS)
- 语义分割 (U-Net/DeepLab)
- 实例分割 (Mask R-CNN)
- 损失函数 (Focal Loss/IoU Loss)
- 注意力机制与特征金字塔

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check_answer(question_num, answer, correct_answer, explanation=""):
    is_correct = answer == correct_answer
    status = "正确!" if is_correct else f"错误! 正确答案是 {correct_answer}"
    print(f"第{question_num}题: {status}")
    if explanation:
        print(f"  解析: {explanation}")
    return is_correct

## 一、选择题部分

---

### 第1题: YOLO 的核心思想

关于 YOLO (You Only Look Once) 目标检测算法，以下说法**错误**的是？

A. 将检测问题转化为回归问题，一次前向传播输出所有边界框

B. YOLO 将输入图像划分为 S x S 的网格，每个网格负责预测中心落在其中的目标

C. YOLO v1 使用了 Anchor Box 来改善对不同尺度目标的检测

D. YOLO 的速度优势来自于不需要生成候选区域 (Region Proposal)

In [ ]:
# 第1题
answer = 'A'  # 修改这里

check_answer(1, answer, 'C', 
    'YOLO v1 没有使用 Anchor Box，这是 YOLOv2 才引入的改进。'
    'YOLOv1 直接预测边界框的坐标和尺寸。从 YOLOv2 开始使用 Anchor Box，'
    '而 YOLOX 和后续版本又回归了 Anchor-free 的设计思路。'
)

print('YOLO 演进简史:')
print('  YOLOv1: 无Anchor, 直接预测坐标, S*S网格')
print('  YOLOv2: 引入Anchor Box, Batch Norm, 多尺度训练')
print('  YOLOv3: Darknet-53, FPN多尺度检测, 3个检测头')
print('  YOLOv4: CSPDarknet, SPP, PANet, Mosaic增强')
print('  YOLOv5: 更好的工程实现, 自动Anchor聚类')
print('  YOLOX:  Anchor-free, Decoupled Head, SimOTA')
print('  YOLOv8: Anchor-free, C2f模块, 分布式焦点损失')

### 第2题: Anchor-based vs Anchor-free 检测

以下关于 Anchor-based 和 Anchor-free 检测方法的对比，**错误**的是？

A. Anchor-based 方法需要预设锚框的大小和比例，通常通过 k-means 聚类数据集得到

B. Anchor-free 方法直接预测目标的位置和尺寸，如 FCOS 预测到边界的距离

C. Anchor-based 方法中每个锚框只需预测偏移量，收敛比 Anchor-free 快

D. Anchor-free 方法不需要调 Anchor 超参数，因此在所有场景下都优于 Anchor-based

In [ ]:
# 第2题
answer = 'A'  # 修改这里

check_answer(2, answer, 'D', 
    'Anchor-free 并非在所有场景下都优于 Anchor-based。两者各有优劣。'
    'Anchor-based 对密集/多尺度目标表现更好，Anchor-free 更简洁但可能需要更多训练技巧。'
)

print('常见方法:')
print('  Anchor-based: YOLOv3-v5, SSD, Faster R-CNN')
print('  Anchor-free: FCOS, CenterNet, YOLOX, YOLOv8')

### 第3题: NMS (非极大值抑制) 的流程

NMS 的标准处理流程，正确顺序是？

A. 计算IoU -> 删除低分框 -> 取最高分 -> 重复

B. 取最高分 -> 计算IoU -> 删除IoU>阈值的框 -> 重复

C. 按置信度排序 -> 取最高分 -> 删除与最高分IoU>阈值的框 -> 重复

D. 删除IoU>阈值的框 -> 按置信度排序 -> 取最高分 -> 重复

In [ ]:
# 第3题
answer = 'A'  # 修改这里

check_answer(3, answer, 'C', 
    'NMS标准流程: 1) 按置信度从高到低排序; '
    '2) 取最高分的框作为保留框; 3) 计算其余框与该框的IoU; '
    '4) 删除IoU超过阈值的框; 5) 在剩余框中重复步骤2-4。'
)

print('NMS变体: Soft-NMS (降低重叠框分数), Softer-NMS (考虑定位不确定性)')

### 第4题: IoU 的计算

框 A = [0, 0, 4, 4] 和框 B = [2, 2, 6, 6] (格式为 [x1, y1, x2, y2])，它们的 IoU 是？

A. 0.14

B. 0.25

C. 0.33

D. 0.50

In [ ]:
# 第4题
answer = 'A'  # 修改这里

check_answer(4, answer, 'A', 
    '交集区域为 [2,2,4,4]，面积 = 2*2 = 4。'
    'A面积=16, B面积=16。并集面积 = 16+16-4 = 28。'
    'IoU = 4/28 = 1/7 = 0.143。'
)

A = np.array([0, 0, 4, 4])
B = np.array([2, 2, 6, 6])
inter_x1 = max(A[0], B[0])
inter_y1 = max(A[1], B[1])
inter_x2 = min(A[2], B[2])
inter_y2 = min(A[3], B[3])
inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
union_area = (A[2]-A[0])*(A[3]-A[1]) + (B[2]-B[0])*(B[3]-B[1]) - inter_area
print(f'IoU = {inter_area}/{union_area} = {inter_area/union_area:.4f}')

### 第5题: mAP 的含义

关于 mAP (mean Average Precision)，以下说法**错误**的是？

A. AP 是 Precision-Recall 曲线下的面积，每个类别一个

B. mAP@0.5 表示 IoU 阈值为 0.5 时计算的 mAP

C. mAP@0.5:0.95 表示 IoU 阈值从 0.5 到 0.95 (步长0.05) 的 mAP 均值

D. mAP 越低越好

In [ ]:
# 第5题
answer = 'A'  # 修改这里

check_answer(5, answer, 'D', 
    'mAP 越高越好! mAP 是目标检测最常用的综合评价指标。'
    'mAP = 所有类别 AP 的平均值。'
    '常用指标: mAP@0.5 (宽松), mAP@0.5:0.95 (严格, COCO标准)'
)

### 第6题: U-Net 的特点

关于 U-Net，以下说法**错误**的是？

A. U-Net 采用编码器-解码器 (Encoder-Decoder) 结构

B. Skip Connection 将编码器的特征图与解码器对应层拼接 (concatenate)

C. U-Net 最初是为医学图像分割设计的，只需要很少的训练样本

D. U-Net 的 skip connection 使用的是逐元素相加 (element-wise add) 而非拼接

In [ ]:
# 第6题
answer = 'A'  # 修改这里

check_answer(6, answer, 'D', 
    'U-Net 的 skip connection 使用的是通道拼接 (concatenate)，不是逐元素相加。'
    '这是 U-Net 和 ResNet 的 skip connection 的区别: '
    'ResNet 用相加，U-Net 用拼接。'
)

### 第7题: FPN (特征金字塔网络)

关于 FPN，以下说法**错误**的是？

A. FPN 通过自顶向下的路径和横向连接融合多尺度特征

B. 高层特征语义信息丰富但分辨率低，低层特征分辨率高但语义信息少

C. FPN 的自顶向下路径使用反卷积进行上采样

D. FPN 可以显著提升小目标的检测性能

In [ ]:
# 第7题
answer = 'A'  # 修改这里

check_answer(7, answer, 'C', 
    'FPN 的自顶向下路径使用的是最近邻上采样，不是反卷积。'
    '然后通过 1x1 卷积 (横向连接) 统一通道数后与底层逐元素相加。'
)

print('FPN 核心思想:')
print('  自底向上(特征提取) + 自顶向下(上采样) + 横向连接(融合)')
print('  使得每层都同时具有丰富的语义信息和高分辨率')

### 第8题: Focal Loss

Focal Loss 的公式是 FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)，其中 gamma 的作用是？

A. 平衡正负样本的数量

B. 降低易分类样本的 loss 权重，让模型聚焦于难样本

C. 控制正则化强度

D. 调整学习率衰减速度

In [ ]:
# 第8题
answer = 'A'  # 修改这里

check_answer(8, answer, 'B', 
    'gamma 是聚焦参数。当样本容易被分类 (p_t 接近1) 时，'
    '(1-p_t)^gamma 接近0，该样本的 loss 被大幅降低。'
    'alpha 用于平衡正负样本比例。'
    '典型参数: gamma=2, alpha=0.25'
)

# 可视化 Focal Loss vs Cross Entropy
p = np.linspace(0.01, 0.99, 100)
ce = -np.log(p)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(p, ce, label='CE (gamma=0)', linewidth=2)
for gamma in [0.5, 1, 2, 5]:
    fl = (1 - p)**gamma * (-np.log(p))
    ax.plot(p, fl, label=f'FL (gamma={gamma})', linewidth=2)

ax.set_xlabel('p_t')
ax.set_ylabel('Loss')
ax.set_title('Focal Loss vs Cross Entropy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 第9题: 数据增强方法

以下哪种数据增强方法会**同时**混合两张图像？

A. Random Crop (随机裁剪)

B. Color Jitter (颜色抖动)

C. Mixup

D. Random Flip (随机翻转)

In [ ]:
# 第9题
answer = 'A'  # 修改这里

check_answer(9, answer, 'C', 
    'Mixup: 将两张图像按比例混合: x_mix = lambda*x1 + (1-lambda)*x2，'
    '标签也按同样比例混合。lambda 从 Beta 分布采样。'
    '其他多图增强: CutMix (裁剪粘贴), Mosaic (4张拼1张)'
)

### 第10题: BatchNorm vs LayerNorm vs GroupNorm

在目标检测等 batch size 较小的任务中，通常推荐使用哪种归一化方法？

A. BatchNorm

B. LayerNorm

C. GroupNorm

D. InstanceNorm

In [ ]:
# 第10题
answer = 'A'  # 修改这里

check_answer(10, answer, 'C', 
    'GroupNorm 将通道分成若干组，每组内计算均值和方差。'
    '它不依赖 batch size，因此在 batch size 较小的任务中表现更好。'
    'BN沿(N,H,W)归一化; LN沿(C,H,W); GN沿(G,H,W); IN沿(H,W)'
)

### 第11题: 转置卷积 (反卷积)

关于转置卷积 (Transposed Convolution)，以下说法**错误**的是？

A. 转置卷积用于上采样，输出尺寸大于输入尺寸

B. 转置卷积也叫反卷积，但严格来说不是数学意义上的反卷积

C. 转置卷积是普通卷积的逆向操作，相当于对输入进行补零后再做卷积

D. 转置卷积不会产生棋盘效应 (Checkerboard Artifacts)

In [ ]:
# 第11题
answer = 'A'  # 修改这里

check_answer(11, answer, 'D', 
    '转置卷积会产生棋盘效应! 当 stride 不能整除 kernel_size 时，'
    '不同位置的像素被覆盖的次数不同，产生规律性的伪影。'
    '缓解方法: 1) 使用 stride=kernel_size; 2) 先上采样再卷积; '
    '3) 使用 PixelShuffle'
)

### 第12题: 注意力机制在CV中的应用

关于注意力机制，以下对应关系**错误**的是？

A. SE-Net: 通道注意力

B. CBAM: 通道注意力 + 空间注意力

C. Vision Transformer (ViT): 自注意力

D. SE-Net 的注意力是加在空间维度上的

In [ ]:
# 第12题
answer = 'A'  # 修改这里

check_answer(12, answer, 'D', 
    'SE-Net 的注意力是加在通道维度上的，不是空间维度。'
    'Squeeze: 全局平均池化压缩空间信息; '
    'Excitation: FC层学习通道依赖关系，输出通道权重。'
)

### 第13题: CIoU / GIoU vs IoU

关于 GIoU、DIoU、CIoU 的改进，按**递进关系**排列正确的是？

A. IoU -> CIoU -> DIoU -> GIoU

B. IoU -> GIoU -> DIoU -> CIoU

C. IoU -> DIoU -> GIoU -> CIoU

D. GIoU -> IoU -> DIoU -> CIoU

In [ ]:
# 第13题
answer = 'A'  # 修改这里

check_answer(13, answer, 'B', 
    '改进递进: IoU -> GIoU -> DIoU -> CIoU。'
    'IoU: 只考虑交集/并集，不相交时梯度为0。'
    'GIoU: 加入最小包围框，解决不相交时无梯度。'
    'DIoU: 加入中心点距离，收敛更快。'
    'CIoU: 在 DIoU 基础上加入宽高比一致性。'
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['IoU: 只看重叠', 'GIoU: +最小包围框', 'DIoU: +中心距离', 'CIoU: +宽高比']
for ax, title in zip(axes, titles):
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect('equal')
    from matplotlib.patches import Rectangle
    gt = Rectangle((2, 2), 4, 4, fill=False, edgecolor='green', linewidth=2, label='GT')
    pred = Rectangle((4, 4), 4, 4, fill=False, edgecolor='red', linewidth=2, label='Pred')
    ax.add_patch(gt); ax.add_patch(pred)
    ax.legend(); ax.set_title(title)
plt.tight_layout()
plt.show()

---

## 二、编程练习部分

---

### 编程题 1: 实现 IoU 计算

**要求**: 
1. 实现单个框的 IoU 计算
2. 实现批量 IoU 计算 (N 个预测框与 M 个真实框)

In [ ]:
def compute_iou(box1, box2):
    """计算两个边界框的 IoU, 格式 [x1, y1, x2, y2]"""
    # TODO
    pass

def compute_iou_batch(boxes1, boxes2):
    """批量计算 IoU, boxes1:(N,4), boxes2:(M,4) -> (N,M)"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def compute_iou(box1, box2):
    """计算两个框的 IoU"""
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0


def compute_iou_batch(boxes1, boxes2):
    """批量 IoU (向量化)"""
    boxes1 = np.array(boxes1, dtype=np.float64)[:, np.newaxis, :]  # (N,1,4)
    boxes2 = np.array(boxes2, dtype=np.float64)[np.newaxis, :, :]  # (1,M,4)
    
    inter_x1 = np.maximum(boxes1[..., 0], boxes2[..., 0])
    inter_y1 = np.maximum(boxes1[..., 1], boxes2[..., 1])
    inter_x2 = np.minimum(boxes1[..., 2], boxes2[..., 2])
    inter_y2 = np.minimum(boxes1[..., 3], boxes2[..., 3])
    
    inter_area = np.maximum(0, inter_x2 - inter_x1) * np.maximum(0, inter_y2 - inter_y1)
    area1 = (boxes1[..., 2] - boxes1[..., 0]) * (boxes1[..., 3] - boxes1[..., 1])
    area2 = (boxes2[..., 2] - boxes2[..., 0]) * (boxes2[..., 3] - boxes2[..., 1])
    union_area = area1 + area2 - inter_area
    
    return np.where(union_area > 0, inter_area / union_area, 0.0)


# 测试
box_a = [0, 0, 4, 4]
box_b = [2, 2, 6, 6]
print(f'单框 IoU: {compute_iou(box_a, box_b):.4f}')

pred = np.array([[10,10,50,50], [30,30,70,70], [200,200,300,300]])
gt = np.array([[12,12,48,48], [300,300,400,400]])
iou_matrix = compute_iou_batch(pred, gt)
print(f'IoU矩阵:\n{iou_matrix.round(4)}')

### 编程题 2: 实现 NMS 算法

**要求**: 实现标准的非极大值抑制算法

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """NMS, boxes:(N,4), scores:(N,) -> keep_indices"""
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def compute_iou_single(box, boxes):
    """一个框与一组框的 IoU"""
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_b = (box[2]-box[0]) * (box[3]-box[1])
    area_bs = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
    union = area_b + area_bs - inter
    return np.where(union > 0, inter / union, 0.0)


def nms(boxes, scores, iou_threshold=0.5):
    """标准贪心 NMS"""
    boxes = np.array(boxes, dtype=np.float64)
    scores = np.array(scores, dtype=np.float64)
    order = scores.argsort()[::-1]
    keep = []
    
    while len(order) > 0:
        i = order[0]
        keep.append(int(i))
        if len(order) == 1:
            break
        ious = compute_iou_single(boxes[i], boxes[order[1:]])
        remaining = np.where(ious <= iou_threshold)[0]
        order = order[remaining + 1]
    
    return keep


# 测试
boxes = np.array([
    [100, 100, 200, 200], [105, 102, 198, 205], [110, 108, 195, 195],
    [300, 300, 400, 400], [310, 305, 405, 398], [500, 500, 600, 600],
], dtype=float)
scores = np.array([0.95, 0.85, 0.75, 0.90, 0.70, 0.60])

keep = nms(boxes, scores, iou_threshold=0.5)
print(f'NMS保留 {len(keep)} 个框:')
for idx in keep:
    print(f'  框{idx}: {boxes[idx].astype(int)}, score={scores[idx]:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = plt.cm.Set3(np.linspace(0, 1, len(boxes)))
for i, (box, score) in enumerate(zip(boxes, scores)):
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                          fill=False, edgecolor=colors[i], linewidth=2)
    axes[0].add_patch(rect)
axes[0].set_title('NMS前')
axes[0].set_xlim(80, 620); axes[0].set_ylim(80, 620)

for i in keep:
    box = boxes[i]
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                          fill=False, edgecolor='green', linewidth=3)
    axes[1].add_patch(rect)
axes[1].set_title(f'NMS后 (保留{len(keep)}个)')
axes[1].set_xlim(80, 620); axes[1].set_ylim(80, 620)
plt.tight_layout()
plt.show()

### 编程题 3: 实现简单的 FPN 结构

**要求**: 用 NumPy 模拟 FPN 的多尺度特征融合过程

In [ ]:
def fpn_forward(features_c2c5, channels=64):
    """
    模拟 FPN 前向传播
    features_c2c5: [C2, C3, C4, C5] 每个 (C_i, H_i, W_i)
    返回: [P2, P3, P4, P5]
    """
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def simple_conv2d(x, weight, bias=None, padding=0):
    """简单 2D 卷积"""
    if padding > 0:
        x = np.pad(x, ((0,0), (padding, padding), (padding, padding)), mode='constant')
    c_out, c_in, kh, kw = weight.shape
    h_out = (x.shape[1] - kh) + 1
    w_out = (x.shape[2] - kw) + 1
    output = np.zeros((c_out, h_out, w_out))
    for co in range(c_out):
        for i in range(h_out):
            for j in range(w_out):
                output[co, i, j] = np.sum(x[:, i:i+kh, j:j+kw] * weight[co])
                if bias is not None:
                    output[co, i, j] += bias[co]
    return output


def fpn_forward(features_c2c5, channels=64):
    """FPN 前向传播模拟"""
    C2, C3, C4, C5 = features_c2c5
    np.random.seed(42)
    
    def make_1x1(c_in, c_out):
        return np.random.randn(c_out, c_in, 1, 1) * 0.01, np.zeros(c_out)
    def make_3x3(c_in, c_out):
        return np.random.randn(c_out, c_in, 3, 3) * 0.01, np.zeros(c_out)
    
    lateral_w = [make_1x1(C.shape[0], channels) for C in [C2, C3, C4, C5]]
    
    M5 = simple_conv2d(C5, *lateral_w[3])
    M4 = M5.repeat(2, axis=1).repeat(2, axis=2)[:,:C4.shape[1],:C4.shape[2]] + simple_conv2d(C4, *lateral_w[2])
    M3 = M4.repeat(2, axis=1).repeat(2, axis=2)[:,:C3.shape[1],:C3.shape[2]] + simple_conv2d(C3, *lateral_w[1])
    M2 = M3.repeat(2, axis=1).repeat(2, axis=2)[:,:C2.shape[1],:C2.shape[2]] + simple_conv2d(C2, *lateral_w[0])
    
    out_w = [make_3x3(channels, channels) for _ in range(4)]
    P2 = simple_conv2d(M2, *out_w[0], padding=1)
    P3 = simple_conv2d(M3, *out_w[1], padding=1)
    P4 = simple_conv2d(M4, *out_w[2], padding=1)
    P5 = simple_conv2d(M5, *out_w[3], padding=1)
    
    return [P2, P3, P4, P5]


# 测试 (使用小尺寸)
np.random.seed(42)
C2 = np.random.randn(64, 16, 16)
C3 = np.random.randn(128, 8, 8)
C4 = np.random.randn(256, 4, 4)
C5 = np.random.randn(512, 2, 2)

P2, P3, P4, P5 = fpn_forward([C2, C3, C4, C5], channels=32)
for name, feat in zip(['P2','P3','P4','P5'], [P2,P3,P4,P5]):
    print(f'{name}: shape={feat.shape}, mean={feat.mean():.4f}')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (name, feat) in enumerate(zip(['C2','C3','C4','C5'], [C2,C3,C4,C5])):
    axes[i].imshow(feat[0], cmap='viridis')
    axes[i].set_title(f'{name} ({feat.shape})')
plt.suptitle('Backbone特征')
plt.tight_layout()
plt.show()

### 编程题 4: 实现 Focal Loss

**要求**: 实现二元分类的 Focal Loss，并与标准交叉熵进行对比

In [ ]:
def focal_loss(predictions, targets, alpha=0.25, gamma=2.0):
    """
    二元 Focal Loss
    predictions: (N,) 概率, targets: (N,) 0或1
    """
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def focal_loss(predictions, targets, alpha=0.25, gamma=2.0):
    """二元 Focal Loss: FL(p_t) = -alpha_t * (1-p_t)^gamma * log(p_t)"""
    p = np.clip(predictions, 1e-7, 1 - 1e-7)
    p_t = np.where(targets == 1, p, 1 - p)
    alpha_t = np.where(targets == 1, alpha, 1 - alpha)
    focal_weight = alpha_t * (1 - p_t) ** gamma
    return np.mean(-focal_weight * np.log(p_t))

def binary_cross_entropy(predictions, targets):
    p = np.clip(predictions, 1e-7, 1 - 1e-7)
    return np.mean(-targets * np.log(p) - (1 - targets) * np.log(1 - p))

# 测试
np.random.seed(42)
N = 1000
targets = np.zeros(N); targets[:100] = 1
predictions = np.random.uniform(0.01, 0.99, N)
predictions[:100] = np.random.uniform(0.6, 0.99, 100)

ce = binary_cross_entropy(predictions, targets)
fl = focal_loss(predictions, targets, alpha=0.25, gamma=2.0)
print(f'Cross Entropy: {ce:.4f}')
print(f'Focal Loss (gamma=2): {fl:.4f}')

# 可视化
p_t = np.linspace(0.01, 0.99, 100)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for gamma in [0, 0.5, 1, 2, 5]:
    fl_curve = -(1 - p_t)**gamma * np.log(p_t)
    label = f'CE' if gamma == 0 else f'FL(g={gamma})'
    axes[0].plot(p_t, fl_curve, label=label, linewidth=2)
axes[0].set_xlabel('p_t'); axes[0].set_ylabel('Loss')
axes[0].set_title('Focal Loss vs CE'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

for gamma in [0, 1, 2, 5]:
    axes[1].plot(p_t, (1-p_t)**gamma, label=f'g={gamma}', linewidth=2)
axes[1].set_xlabel('p_t'); axes[1].set_ylabel('(1-p_t)^gamma')
axes[1].set_title('调制因子'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 编程题 5: 实现 Grad-CAM (类激活图可视化)

**要求**: 实现简化的 Grad-CAM 算法

Grad-CAM 流程:
1. 前向传播获取目标层的特征图 A (C x H x W)
2. 反向传播获取目标类别对特征图的梯度
3. 对梯度在空间维度做全局平均池化得到权重
4. 加权求和 + ReLU + 归一化

In [ ]:
def grad_cam(feature_maps, gradients):
    """
    Grad-CAM
    feature_maps: (C, H, W), gradients: (C, H, W)
    返回: cam (H, W), 值在 [0, 1]
    """
    # TODO
    pass

In [ ]:
# ====== 参考答案 ======

def grad_cam(feature_maps, gradients):
    """Grad-CAM 实现"""
    # 全局平均池化梯度 -> 通道权重
    alpha = gradients.mean(axis=(1, 2))  # (C,)
    
    # 加权求和
    cam = np.einsum('c,cij->ij', alpha, feature_maps)  # (H, W)
    
    # ReLU + 归一化
    cam = np.maximum(cam, 0)
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam


def visualize_gradcam(image, cam):
    """可视化 Grad-CAM 热力图"""
    from scipy.ndimage import zoom
    if cam.shape != image.shape[:2]:
        factors = (image.shape[0]/cam.shape[0], image.shape[1]/cam.shape[1])
        cam = zoom(cam, factors, order=1)
    heatmap = plt.cm.jet(cam)[:,:,:3]
    img_rgb = image.copy()
    if img_rgb.max() > 1:
        img_rgb = img_rgb / 255.0
    return np.clip(0.6 * img_rgb + 0.4 * heatmap, 0, 1), cam


# 测试
np.random.seed(42)
h, w, C = 16, 16, 64

feature_maps = np.random.randn(C, h, w) * 0.1
feature_maps[:, 4:10, 4:12] += np.random.uniform(1, 3, (C, 6, 8))

gradients = np.random.randn(C, h, w) * 0.1
gradients[:, 4:10, 4:12] += np.random.uniform(0.5, 2, (C, 6, 8))

cam = grad_cam(feature_maps, gradients)

img_h, img_w = 128, 128
test_image = np.random.rand(img_h, img_w) * 0.3
test_image[32:80, 32:96] += 0.5
test_image = np.clip(test_image, 0, 1)

overlay, cam_resized = visualize_gradcam(test_image, cam)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(test_image, cmap='gray'); axes[0].set_title('输入图像')
axes[1].imshow(cam, cmap='jet'); axes[1].set_title(f'Grad-CAM ({h}x{w})')
axes[2].imshow(overlay); axes[2].set_title(f'叠加结果 ({img_h}x{img_w})')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

---

## 三、面试要点总结

| 主题 | 关键要点 |
|------|----------|
| YOLO | 单次前向传播; v1无Anchor, v2引入Anchor, X/v8回归Anchor-free |
| Anchor | Anchor-based需预设框+预测偏移; Anchor-free直接预测位置 |
| NMS | 按分数排序->取最高->删IoU>阈值的->重复; Soft-NMS降低分数而非删除 |
| IoU系列 | IoU->GIoU(+包围框)->DIoU(+中心距离)->CIoU(+宽高比) |
| mAP | 各类AP均值; mAP@0.5宽松, mAP@0.5:0.95严格(COCO标准) |
| U-Net | 编码器-解码器+skip connection(拼接); 医学图像分割经典 |
| FPN | 自顶向下+横向连接(1x1卷积+逐元素相加); 多尺度特征融合 |
| Focal Loss | 降低易样本权重, 聚焦难样本; gamma控制聚焦程度 |
| 归一化 | BN依赖batch size; GN分组不依赖; 检测/分割推荐GN |
| 转置卷积 | 上采样, 可能产生棋盘效应; 可用双线性插值+普通卷积替代 |
| 注意力 | SE-Net(通道), CBAM(通道+空间), ViT(自注意力/patch) |
| 数据增强 | Mixup/CutMix/Mosaic是多图增强; 裁剪/翻转/颜色抖动是单图增强 |